# <a id='toc1_'></a>[SARIMAX](#toc0_)

**Table of contents**<a id='toc0_'></a>    
- [SARIMAX](#toc1_)    
  - [Import + setup](#toc1_1_)    
  - [Upload](#toc1_2_)    
  - [Calculation of Relevance Score](#toc1_3_)    
  - [Selection of exogenous](#toc1_4_)    
  - [Optimization of p and q](#toc1_5_)    
  - [Loading of integration orders](#toc1_6_)    
  - [Main flow for selection of best-scored exogenous and best-order ARIMAX for that model](#toc1_7_)    
  - [Recursive forecast for actual forecasting](#toc1_8_)    
  - [Future forecasts 2030](#toc1_9_)    

<!-- vscode-jupyter-toc-config
	numbering=false
	anchor=true
	flat=false
	minLevel=1
	maxLevel=6
	/vscode-jupyter-toc-config -->
<!-- THIS CELL WILL BE REPLACED ON TOC UPDATE. DO NOT WRITE YOUR TEXT IN THIS CELL -->

<a class="anchor" id="upload"> </a>
## <a id='toc1_1_'></a>[Import + setup](#toc0_)

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from src.config import set_seeds
import src.pipeline as pipe
import src.models as mod
import src.evaluation as eval
import src.reporting as rep
import src.visualization as visual
import numpy as np
import pandas as pd
import itertools
import ast
import re
import os
import warnings
warnings.filterwarnings('ignore')
import statsmodels.api as sm
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import grangercausalitytests
from sklearn.feature_selection import mutual_info_regression

In [3]:
set_seeds()

Random seeds set to 42. Deterministic operations enabled.


In [6]:
CATEGORIZED_VARS = {
    'demography': [
        'population_percent',
        'population_growth',
        'population_abs',
        'employment_tot',
        'employment_male',
        'employment_female'
    ],
    'land_use': [
        'forestarea_percent',
        'forestarea_abs',
        'agriland_percent',
        'agriland_abs',
        'arableland_percent',
        'arableland_person',
        'arableland_abs',
        'cerealland_abs',
        'cropland_percent'
    ],
    'prerequisites': [
        'withdrawals_percent',
        'fertilizer_abs',
        'fertilizer_percent'
    ],
    'production': [
        'livestock_production_index',
        'food_production_index',
        'crop_production_index',
        'cereal_production',
        'cerealyield_abs'
    ],
    'economy': [
        'valueadded_percent',
        'valueadded_dollars',
        'exports_percent',
        'imports_percent'
    ]
}

## <a id='toc1_2_'></a>[Upload](#toc0_)

In [7]:
df = pipe.load_data()
display(df.head())

Dropped 9 unusable indicators.
Dataset loaded: 65 years (from 1960 to 2024), 27 variables.


,population_percent,population_growth,population_abs,employment_tot,employment_male,employment_female,forestarea_percent,forestarea_abs,agriland_percent,agriland_abs,...,fertilizer_percent,livestock_production_index,food_production_index,crop_production_index,cereal_production,cerealyield_abs,valueadded_percent,valueadded_dollars,exports_percent,imports_percent
1960-01-01,40.639,NaN,20400656.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1961-01-01,40.144,-0.557139,20287312.0,NaN,NaN,NaN,NaN,NaN,70.324028,206830.0,...,72.707716,70.91,85.93,93.68,13933400.0,2181.5,NaN,NaN,NaN,NaN
1962-01-01,39.645,-0.574190,20171158.0,NaN,NaN,NaN,NaN,NaN,70.218626,206520.0,...,70.071359,72.59,86.90,94.43,14433210.0,2225.3,NaN,NaN,2.563660,16.566057
1963-01-01,39.147,-0.534554,20063620.0,NaN,NaN,NaN,NaN,NaN,69.735813,205100.0,...,63.883735,65.95,86.37,97.19,13324660.0,2115.2,NaN,NaN,2.651714,14.571604
1964-01-01,38.650,-0.455075,19972523.0,NaN,NaN,NaN,NaN,NaN,69.572609,204620.0,...,64.593354,69.57,90.28,101.31,14007520.0,2243.0,NaN,NaN,2.792923,15.115329


## <a id='toc1_3_'></a>[Calculation of Relevance Score](#toc0_)

In [8]:
def calculate_relevance_score(target, candidate, wide_df):
    """
    Calcola rilevanza usando test statistici robusti
    Restituisce punteggio 0-1 basato su evidenza statistica
    """
    try:
        data = wide_df[[target, candidate]].dropna()
        if len(data) < 15:
            return 0
        
        scores = []
        
        # 1. CORRELAZIONE (25%)
        correlation = abs(data.corr().iloc[0, 1])
        if not np.isnan(correlation):
            scores.append(('correlation', correlation * 0.25))

        # 2. GRANGER CAUSALITY (35%) = verifica se X aiuta a prevedere Y
        def granger_causality_test(target, candidate, wide_df, maxlag=2):
            try:
                data = wide_df[[target, candidate]].dropna()
                if len(data) < 20:
                    return 0

                test_result = grangercausalitytests(data, maxlag=maxlag, verbose=False)
                # Prendi il p-value minimo tra tutti i lag
                min_pvalue = min([test_result[lag][0]['ssr_ftest'][1] for lag in range(1, maxlag+1)])
                
                # P-value basso = causalità Granger significativa
                if min_pvalue < 0.05:
                    return 1.0  # Relazione forte
                elif min_pvalue < 0.1:
                    return 0.7  # Relazione moderata
                else:
                    return 0.3  # Relazione debole
            except:
                return 0

        granger_score = granger_causality_test(target, candidate, wide_df)
        scores.append(('granger', granger_score * 0.35))
        
        # 3. MUTUAL INFORMATION (25%) = misura dipendenze non-lineari
        def mutual_information_score(target, candidate, wide_df):
            try:
                data = wide_df[[target, candidate]].dropna()
                if len(data) < 15:
                    return 0
                
                X = data[candidate].values.reshape(-1, 1)
                y = data[target].values
                mi = mutual_info_regression(X, y, random_state=42)[0]
                
                # Normalizza tra 0 e 1
                return min(mi * 10, 1.0)
            except:
                return 0
        
        mi_score = mutual_information_score(target, candidate, wide_df)
        scores.append(('mutual_info', mi_score * 0.25))
        
        # 4. STABILITY - numero di osservazioni (15%)
        stability_score = min(len(data) / 50, 1.0)  # Normalizza su 50 osservazioni
        scores.append(('stability', stability_score * 0.15))
        
        # Punteggio totale
        total_score = sum(score for _, score in scores)
        return min(total_score, 1.0)
        
    except Exception as e:
        print(f"Errore nel calcolo rilevanza {target}-{candidate}: {e}")
        return 0

## <a id='toc1_4_'></a>[Selection of exogenous](#toc0_)

In [17]:
def select_one_per_category(target, candidates):
    target_category = next((cat for cat, vars in CATEGORIZED_VARS.items() if target in vars), 'other')
    selected_exog = []
    used_categories = set()

    if target_category:
        used_categories.add(target_category)  
    
    for c in candidates:
        if c['category'] in used_categories:
            continue
        selected_exog.append(c)
        used_categories.add(c['category'])
    
    return selected_exog

In [ ]:
def select_exogenous(wide_df, target):
    candidates_exog = []
    for col in wide_df.columns:
        if col == target: continue
        if col == 'Year': continue
        candidates_exog.append({
            'name': col,
            'category': next((cat for cat, vars in CATEGORIZED_VARS.items() if col in vars), 'other'),
            'relevance_score': calculate_relevance_score(target, col, wide_df),
            'data_points': wide_df[col].notna().sum()
        })
    
    # ordino per rilevanza così mi prende la prima esogena per rilevanza di ogni categoria
    candidates_exog.sort(key=lambda x: x['relevance_score'], reverse=True)
    # print('prime 5 per rilevanze')
    # for c in candidates_exog[:5]:
    #     print(f"{c['name']}: {c['category']}")
    # print('\n')
    
    selected = select_one_per_category(target, candidates_exog)
    return selected

## <a id='toc1_5_'></a>[Optimization of p and q](#toc0_)

In [24]:
def optimize_SARIMAX(target, exog, d):   
    best_aic = float('inf')
    best_order = None
    results = []
    
    for p, q in itertools.product(range(0, 7), 
                                range(0, 7)):
        if p == 0 and q == 0: continue
            
        try:
            model = SARIMAX(target,
                        exog=exog,
                        order=(p, d, q),
                        seasonal_order=(0, 0, 0, 0),
                        enforce_stationarity=False,
                        enforce_invertibility=False)
            fitted_model = model.fit(disp=False, maxiter=200)
            
            results.append({
                'order': (p, d, q),
                'aic': fitted_model.aic,
                'converged': True
            })
            
            if fitted_model.aic < best_aic:
                best_aic = fitted_model.aic
                best_order = (p, d, q)
                
        except Exception as e:
            results.append({
                'order': (p, d, q),
                'aic': np.nan,
                'converged': False,
                'error': str(e)
            })
            continue
    
    # Ordina risultati per AIC
    valid_results = [r for r in results if r['converged']]
    valid_results.sort(key=lambda x: x['aic'])
    
    return {
            'target': target,
            'best_order': best_order,
            'best_aic': best_aic,
            'valid_results': valid_results
        }

In [25]:
# DEBUG: verifica che tutto sia allineato
def debug_alignment(target, exog):
    print(f"{'/'*30} VERIFICA ALLINEAMENTO: {'/'*30}")
    print(f"   Target series length: {len(target)}")
    print(f"   Exog data shape: {exog.shape}")
    print(f"   Target index: {target.index[:5].tolist()}")
    print(f"   Exog index: {exog.index[:5].tolist()}")
    
    # Verifica che gli indici siano identici
    indices_match = target.index.equals(exog.index)
    print(f"Indici allineati: {indices_match}")

## <a id='toc1_6_'></a>[Loading of integration orders](#toc0_)

In [21]:
config_path = "../results/integration_orders.xlsx"
if os.path.exists(config_path):
    print(f"Loading configuration from: {config_path}")
    config_df = pd.read_excel(config_path)
    if 'Indicator' in config_df.columns:
        config_df.set_index('Indicator', inplace=True)
    print("Configuration loaded correctly.")
else:
    print(f"ERROR: file {config_path} not found.")

Loading configuration from: ../results/integration_orders.xlsx
Configuration loaded correctly.


## <a id='toc1_7_'></a>[Main flow for selection of best-scored exogenous and best-order ARIMAX for that model](#toc0_)

In [ ]:
df_orders = pd.DataFrame(columns=["Target", "Best order", "Best AIC", "N_Exog", "Exog_Used"])

for target in df.columns:
    if not np.issubdtype(df[target].dtype, np.number): continue
    if df[target].dropna().empty: continue

    try:
        d_train = int(config_df.loc[target, 'd_train'])
        d_full  = int(config_df.loc[target, 'd_full'])
    except KeyError:
        print(f"Skipping {target}: No integration order found in config.")
        continue
    
    selected_exog = select_exogenous(df, target)
    print(selected_exog)
    if selected_exog and isinstance(selected_exog, list):
        if isinstance(selected_exog[0], dict): exog_names = [exog['name'] for exog in selected_exog]
        else: exog_names = selected_exog
    else: exog_names = []
    
    if exog_names:
        data_subset = df[[target] + exog_names].dropna().copy()
        data_subset['Year'] = data_subset.index.year
        data_subset = data_subset.reset_index(drop=True)
    else: 
        print("No valid exogenous")
        continue
    if len(data_subset) < 20:
        print("Not enough data after alignment")
        continue
    
    # resoconto
    print(f"{'-'*30} TARGET: {target.upper()} {'-'*30}")
    print(f"Numero di differenziazioni necessarie a rendere stazionaria la serie: {d_train}")
    print("Esogene selezionate:")
    for i, e in enumerate(selected_exog, 1):
        if isinstance(e, dict):
            name = e.get('name', '')
            cat = e.get('category', 'other')
            score = e.get('relevance_score', np.nan)
            print(f"  {i:2d}. {name} — {cat} — rilevanza: {score:.3f}")
        else:
            print(f"  {i:2d}. {e}")
    print(f"Periodo: {data_subset['Year'].min()}-{data_subset['Year'].max()}")
    print(f"Osservazioni: {len(data_subset)}")
    optimized = (optimize_SARIMAX(data_subset[target], data_subset[exog_names], d_train))
    if optimized:
        best_order = optimized.get('best_order')
        best_aic = optimized.get('best_aic', np.nan)
    else:
        best_order = None
        best_aic = np.nan

    new_row = pd.DataFrame({
        'Target': [target],
        'Best order': [best_order],
        'Best AIC': [best_aic],
        'N_Exog': [len(exog_names)],
        'Exog_Used': [exog_names]
    })
    df_orders = pd.concat([df_orders, new_row], ignore_index=True)
    print(f"Salvato: Target = {target},")
    print(f"--> Best order = {best_order},")
    print(f"--> Best AIC = {best_aic},")
    print(f"--> N_Exog = {len(exog_names)},")
    print(f"--> Exog used = {exog_names}")
    # debug_alignment(data_subset[target], data_subset[exog_names])
print(df_orders)

prime 5 per rilevanze
arableland_person: land_use
arableland_abs: land_use
cerealland_abs: land_use
agriland_abs: land_use
agriland_percent: land_use


[{'name': 'arableland_person', 'category': 'land_use', 'relevance_score': 0.9891006683417826, 'data_points': 61}, {'name': 'imports_percent', 'category': 'economy', 'relevance_score': 0.9835035702349705, 'data_points': 63}, {'name': 'cerealyield_abs', 'category': 'production', 'relevance_score': 0.9700035345722743, 'data_points': 62}, {'name': 'withdrawals_percent', 'category': 'prerequisites', 'relevance_score': 0.8553423864873496, 'data_points': 52}]
------------------------------ TARGET: POPULATION_PERCENT ------------------------------
Numero di differenziazioni necessarie a rendere stazionaria la serie: 2
Esogene selezionate:
   1. arableland_person — land_use — rilevanza: 0.989
   2. imports_percent — economy — rilevanza: 0.984
   3. cerealyield_abs — production — rilevanza: 0.970
   4. withdrawals_percent — prerequisites — rileva

## <a id='toc1_8_'></a>[Recursive forecast for actual forecasting](#toc0_)

In [ ]:
def recursive_forecast(endog, exog, train_len, horizon, window, method, p, d, q) -> list:
    total_len = len(endog)
    
    if method == 'last':
        pred_last_value = []
        for i in range(train_len, total_len, window):
            if i == 0: last_value = 0
            else: last_value = endog[:i].iloc[-1]
            steps = min(window, total_len - i)
            pred_last_value.extend([last_value] * steps)
        return pred_last_value[:horizon]
    
    elif method == 'ARIMAX':
        pred_ARIMAX = []
        for i in range(train_len, total_len, window):
            try:
                # Addestra sul dati fino a i
                model = SARIMAX(endog[:i], exog=exog[:i] if exog is not None else None, 
                            order=(p, d, q), seasonal_order=(0, 0, 0, 0), 
                            simple_differencing=False)
                res = model.fit(disp=False)
                
                # Forecast per i prossimi 'window' passi
                steps = min(window, total_len - i)
                
                if exog is not None:
                    # Usa le esogene FUTURE per il forecast
                    exog_forecast = exog.iloc[i:i+steps]
                    forecast = res.get_forecast(steps=steps, exog=exog_forecast)
                else:
                    forecast = res.get_forecast(steps=steps)
                
                pred_ARIMAX.extend(forecast.predicted_mean.values)
                
            except Exception as e:
                print(f"ARIMAX failed at step {i}: {e}")
                last_value = endog[:i].iloc[-1] if i > 0 else 0
                steps = min(window, total_len - i)
                pred_ARIMAX.extend([last_value] * steps)
        
        return pred_ARIMAX[:horizon]

In [26]:
all_results = {}
for target in df.columns:
    if not np.issubdtype(df[target].dtype, np.number): continue
    if df[target].dropna().empty: continue

    print(f"{'-'*30} {target.upper()} {'-'*30}")
    try:
        d_train = int(config_df.loc[target, 'd_train'])
        d_full  = int(config_df.loc[target, 'd_full'])
    except KeyError:
        print(f"Skipping {target}: No integration order found in config.")
        continue
    
    try:
        # prendo riga con cose che ho calcolato di quel target
        row = df_orders[df_orders["Target"] == target]
        if row.empty:
            print(f"Target '{target}' not found in df_orders")
            continue
        
        # riprendo p d q
        best_order_val = row['Best order'].iloc[0]
        if isinstance(best_order_val, str):
            order = ast.literal_eval(best_order_val)
        else:
            order = tuple(best_order_val)
        p, d, q = map(int, order)
        
        # riprendo esogene ottimizzate
        exog_val = row['Exog_Used'].iloc[0]
        if not isinstance(exog_val, list):
            exog_val = [exog_val] if exog_val else []

        # filtro solo colonne che mi servono: target e esogene
        keep_cols = [target] + [c for c in exog_val if c in df.columns]
        data_subset = df[keep_cols].dropna().copy()
        data_subset['Year'] = data_subset.index.year
        data_subset = data_subset.reset_index(drop=True)
        final_cols = ['Year'] + keep_cols
        final_cols = list(dict.fromkeys(final_cols))
        data_subset = data_subset[final_cols]
        if len(data_subset) < 20:
            print(f"Dati insufficienti dopo il dropna() ({len(data_subset)} righe)")
            continue
        
        # divido in train e test set
        TRAIN_LEN = int(0.8 * len(data_subset))
        TOTAL_LEN = len(data_subset)
        HORIZON = TOTAL_LEN - TRAIN_LEN
        WINDOW = 1
        
        # endogena = target, esogene = quelle calcolate prima
        endog = data_subset[target]
        exog = data_subset[exog_val] if exog_val else None

        pred_ARIMAX = recursive_forecast(
            endog, exog, 
            train_len=TRAIN_LEN, 
            horizon=HORIZON, 
            window=WINDOW, 
            method='ARIMAX', 
            p=p, d=d, q=q
        )

        pred_last_value = recursive_forecast(
            endog, None,
            train_len=TRAIN_LEN, 
            horizon=HORIZON, 
            window=WINDOW, 
            method='last', 
            p=p, d=d, q=q
        )

        results_df = data_subset.iloc[TRAIN_LEN:].copy()
        results_df = results_df.rename(columns={target: 'Value'})
        results_df['pred_ARIMAX'] = pred_ARIMAX
        results_df['pred_last_value'] = pred_last_value
        
        # salvo df risultati in dizionario con tutti
        all_results[target] = results_df
        print(results_df[['Year', 'Value', 'pred_ARIMAX', 'pred_last_value']].head())

    except Exception as e:
        print(f"Error processing {target}: {e}")
        continue

------------------------------ POPULATION_PERCENT ------------------------------
    Year   Value  pred_ARIMAX  pred_last_value
41  2011  31.556    31.528101           31.673
42  2012  31.316    31.499812           31.556
43  2013  31.021    31.010385           31.316
44  2014  30.728    30.723820           31.021
45  2015  30.435    30.433267           30.728
------------------------------ POPULATION_GROWTH ------------------------------
    Year     Value  pred_ARIMAX  pred_last_value
26  2016 -1.147508    -1.196861        -1.109249
27  2017 -1.151428    -1.081395        -1.147508
28  2018 -0.745903    -1.168875        -1.151428
29  2019 -1.713177    -0.658447        -0.745903
30  2020 -1.527891    -1.595364        -1.713177
------------------------------ POPULATION_ABS ------------------------------
    Year       Value   pred_ARIMAX  pred_last_value
48  2010  18946601.0  1.899312e+07       18933274.0
49  2011  18942070.0  1.874499e+07       18946601.0
50  2012  18849491.0  1.894654

In [27]:
if 'all_results' not in globals() or not all_results:
    print("Dizionario 'all_results' non trovato o vuoto.")
else:
    for target, results_df in all_results.items():
        try:
            row = df_orders[df_orders["Target"] == target]
            if row.empty:
                print(f"Skipping {target}: Config not found in df_orders.")
                continue
            
            # Parsing order
            best_order_val = row['Best order'].iloc[0]
            if isinstance(best_order_val, str):
                order = ast.literal_eval(best_order_val)
            else:
                order = tuple(best_order_val)
            p, d, q = map(int, order)
            
            # Parsing Exog
            exog_val = row['Exog_Used'].iloc[0]
            if not isinstance(exog_val, list):
                exog_val = [exog_val] if exog_val and pd.notna(exog_val) else []

            if 'Year' not in df.columns:
                df['Year'] = df.index.year
            
            keep_cols = [target] + [c for c in exog_val if c in df.columns]
            keep_cols = ['Year'] + keep_cols
            keep_cols = list(dict.fromkeys(keep_cols))
            data_subset = df[keep_cols].dropna().reset_index(drop=True)
            
            TRAIN_LEN = int(0.8 * len(data_subset))
            train_data = data_subset.iloc[:TRAIN_LEN]
            
            test_data = results_df.copy()
            if test_data.empty or train_data.empty:
                print(f"Skipping {target}: Empty train or test data.")
                continue

            print(f"Processing {target}...")
            pred_metrics = eval.compute_errors(test_data['Value'], test_data['pred_ARIMAX'])
            pred_residuals = eval.compute_residual_diagnostics(test_data['Value'], test_data['pred_ARIMAX'])
            
            print(f"  -> Residuals: Mean={pred_residuals['residual_mean']:.4f}, "
                f"Shapiro P={pred_residuals['shapiro_wilk_pvalue']:.4f}, "
                f"Ljung-Box P={pred_residuals['ljung_box_pvalue']:.4f}")
            
            train_plot = train_data.rename(columns={target: 'Value'}).set_index('Year')
            train_plot = train_plot[['Value']]
            test_plot = test_data.set_index('Year')
            test_plot = test_plot[['Value']]
            pred_arimax_plot = test_data.set_index('Year')['pred_ARIMAX']
            pred_last_plot = test_data.set_index('Year')['pred_last_value']
            
            visual.plot_forecast(
                train=train_plot,
                test=test_plot,
                variable_name=target,
                model_name="ARIMAX",
                folder_name="05_ARIMAX",
                prediction=pred_arimax_plot,
                baseline=pred_last_plot,
                baseline_name="Last Value",
                rmse=pred_metrics['RMSE'],
                save_plot=True
            )
    
            lb_p = pred_residuals['ljung_box_pvalue']
            sw_p = pred_residuals['shapiro_wilk_pvalue']
            
            # Se p > 0.05 è buono (OK), altrimenti è (NO)
            lb_status = "OK" if lb_p > 0.05 else "NO"
            sw_status = "OK" if sw_p > 0.05 else "NO"
            
            plot_title_suffix = f"\nLjung-Box p={lb_p} ({lb_status}) | Shapiro p={sw_p} ({sw_status})"
            
            visual.plot_residuals(
                y_true=test_data['Value'],
                y_pred=test_data['pred_ARIMAX'],
                variable_name=target,
                model_name=f"ARIMAX",
                folder_name="05_residARIMAX"
            )
    
            config_str = f"p={p}_d={d}_q={q}_exog={len(exog_val)}"
            rep.save_experiment_results(
                indicator=target,
                model_name='ARIMAX',
                configuration=config_str,
                y_test=test_data['Value'].values,
                y_pred=test_data['pred_ARIMAX'].values,
                years_test=test_data['Year'].values,
                y_train=train_data[target].values,
                params={'p': p, 'd': d, 'q': q, 'exog': exog_val},
                training_time=None
            )
        except Exception as e:
            print(f"Error saving/plotting {target}: {e}")
            continue

Processing population_percent...
  -> Residuals: Mean=-0.0140, Shapiro P=0.0000, Ljung-Box P=0.5201
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_ARIMAX\ARIMAX_population_percent.png
Residuals saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_residARIMAX\residARIMAX_population_percent.png
Saving results for ARIMAX | population_percent...
Leaderboard updated: population_percent | ARIMAX
Save leaderboard complete.
Processing population_growth...
  -> Residuals: Mean=-0.0649, Shapiro P=0.0399, Ljung-Box P=0.1273
Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_ARIMAX\ARIMAX_population_growth.png
Residuals saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_residARIMAX\residARIMAX_population_growth.png
Saving results for ARIMAX | population_growth...
Leaderboard updated: population_growth | ARIMAX
Save leaderboard complete.
Processing population_a

## <a id='toc1_9_'></a>[Future forecasts 2030](#toc0_)

In [ ]:
BASELINE_NAME = "NAIVE_LAST"
MODEL_NAME = "ARIMAX"

for target in df.columns:
    if target not in all_results:
        continue
        
    print(f"\nProcessing: {target}...")
    try:
        row = df_orders[df_orders["Target"] == target]
        if row.empty:
            print(f"Skipping {target}: No order found in df_orders.")
            continue

        best_order_val = row['Best order'].iloc[0]
        exog_val = row['Exog_Used'].iloc[0]

        if not isinstance(exog_val, list):
            exog_val = [exog_val] if exog_val and pd.notna(exog_val) else []

        if isinstance(best_order_val, str):
            order = ast.literal_eval(best_order_val)
        else:
            order = tuple(best_order_val)
        
        wide_df = df.copy()
        
        if 'Year' not in wide_df.columns and hasattr(wide_df.index, 'year'):
            wide_df['Year'] = wide_df.index.year
        
        keep_cols = [target] + [c for c in exog_val if c in wide_df.columns]
        if 'Year' not in keep_cols:
            keep_cols = ['Year'] + keep_cols
        
        data_subset = wide_df[keep_cols].dropna().reset_index(drop=True)
        
        endog = data_subset[target]
        exog = data_subset[exog_val] if exog_val else None

        if endog.empty:
            print(f"No data for {target}. Skipping.")
            continue
            
        model = sm.tsa.SARIMAX(
            endog,
            exog=exog,
            order=order,
            seasonal_order=(0, 0, 0, 0),
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        results = model.fit(disp=False, maxiter=200)
        
        last_year = data_subset['Year'].max()
        forecast_years = pd.date_range(start=f'{int(last_year)+1}-01-01', end='2030-01-01', freq='YS')
        forecast_steps = len(forecast_years)

        if exog is not None:
            last_exog_values = exog.iloc[-1]
            exog_forecast = pd.DataFrame(
                [last_exog_values] * forecast_steps,
                columns=exog.columns,
                index=forecast_years 
            )
            forecast_res = results.get_forecast(steps=forecast_steps, exog=exog_forecast)
        else:
            forecast_res = results.get_forecast(steps=forecast_steps)

        forecast_mean = forecast_res.predicted_mean
        forecast_ci = forecast_res.conf_int()

        future_pred_df = pd.DataFrame({
            'year': forecast_years.year,
            'pred': forecast_mean.values,
            'lower_ci': forecast_ci.iloc[:, 0].values,
            'upper_ci': forecast_ci.iloc[:, 1].values
        })

        last_observed_val = endog.iloc[-1]
        baseline_pred_df = pd.DataFrame({
            'year': forecast_years.year,
            'pred': [last_observed_val] * forecast_steps
        })
        
        full_history_series = endog.copy()
        full_history_series.index = data_subset['Year']

        visual.plot_future_forecasts(
            full_history=full_history_series,
            future_pred=future_pred_df,
            baseline_pred=baseline_pred_df,
            variable_name=target,
            model_name=MODEL_NAME,
            baseline_name=BASELINE_NAME,
            folder_name="05_futureARIMAX",
            save_plots=True
        )
    except Exception as e:
        print(f"Error processing {target}: {e}")


Processing: population_percent...
Cartella creata: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_futureARIMAX
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_futureARIMAX\futureARIMAX_population_percent.png

Processing: population_growth...
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_futureARIMAX\futureARIMAX_population_growth.png

Processing: population_abs...
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_futureARIMAX\futureARIMAX_population_abs.png

Processing: employment_tot...
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_futureARIMAX\futureARIMAX_employment_tot.png

Processing: employment_male...
Future Plot saved: C:\Users\oldan\Desktop\RuralDevelopment\progetto-tirocinio\results\plots\05_futureARIMAX\futureARIMAX_employment_male.png

Processing: employment